In [ ]:
!bash download_data.sh

In [ ]:
import pandas as pd
import numpy as np

def generate_bed(input_bed, output_bed, flank=2048, seed=42):
    np.random.seed(seed)  # reproducible randomness
    
    cols = ["chrom", "start", "end", "id1", "id2", "label"]
    df = pd.read_csv(input_bed, sep="\t", header=None, names=cols)

    out_rows = []
    for _, row in df.iterrows():
        start, end = int(row["start"]), int(row["end"])
        center = (start + end) / 2

        # define central 50% of the region
        quarter = (end - start) * 0.25
        low = start + quarter
        high = end - quarter

        # pick random center inside that range
        chosen_center = np.random.randint(low, high + 1)

        # define 4096bp window
        new_start = int(chosen_center - flank)
        new_end = int(chosen_center + flank)

        out_rows.append([row["chrom"], new_start, new_end])

    out_df = pd.DataFrame(out_rows, columns=["chrom", "start", "end"])
    out_df.to_csv(output_bed, sep="\t", header=False, index=False)

# Example usage
generate_bed("GRCh38-cCREs.bed", "GRCh38_cCREs_4kb.bed")


In [14]:
import os
import pandas as pd

def convert_tsv_to_vcf(folder_path):
    for filename in os.listdir(folder_path):
       
        file_path = os.path.join(folder_path, filename)

        try:
            df = pd.read_csv(file_path, sep='\t')
        except Exception as e:
            print(f"Could not read {filename}: {e}")
            continue

        required_cols = ["chrom", "pos", "strand", "ref", "alt"]
        if not all(col in df.columns for col in required_cols):
            print(f"Skipping {filename}: missing required columns.")
            continue

        df_vcf = df[required_cols].copy()

        # Convert strand from 1/-1 to +/-
        df_vcf["strand"] = df_vcf["strand"].map({1: "+", -1: "-"})
        if df_vcf["strand"].isnull().any():
            print(f"Warning: Unrecognized strand value in {filename}")

        # Rename columns
        df_vcf.columns = ["CHROM", "POS", "STRAND", "REF", "ALT"]
        df_vcf["STRAND"] = "+" # the strand info here already is for the gene, not the variant

        # Output file path
        base_name = os.path.splitext(filename)[0]
        output_path = os.path.join(folder_path, f"{base_name}.vcf")

        df_vcf.to_csv(output_path, sep='\t', index=False, header=False)
        print(f"Converted {filename} -> {base_name}.vcf")

convert_tsv_to_vcf("../data_test/")


Converted MPRA_eQTL.tsv -> MPRA_eQTL.vcf
Converted MPRA_saturation.tsv -> MPRA_saturation.vcf
Converted CAGI5_saturation.tsv -> CAGI5_saturation.vcf
Skipping GEL_RNA.vcf: missing required columns.
Skipping GTEx_outlier.vcf: missing required columns.
Skipping UKBB_proteome.vcf: missing required columns.
Converted GTEx_eQTL.tsv -> GTEx_eQTL.vcf
Skipping index.html: missing required columns.
Converted finetune_gtex.tsv -> finetune_gtex.vcf
Skipping finetune_gtex.vcf: missing required columns.
Skipping MPRA_eQTL.vcf: missing required columns.
Skipping CAGI5_saturation.vcf: missing required columns.
Converted GEL_RNA.tsv -> GEL_RNA.vcf
Skipping MPRA_saturation.vcf: missing required columns.
Converted GTEx_outlier.tsv -> GTEx_outlier.vcf
Converted UKBB_proteome.tsv -> UKBB_proteome.vcf
Skipping GTEx_eQTL.vcf: missing required columns.
